# 🎯 Guia Completo — Cientista de Dados Itaú

**Autor:** Plano de estudo personalizado  
**Objetivo:** Passar na sabatina de Cientista de Dados do Itaú Unibanco  

---

## Estrutura do Notebook

| Dia | Tema |
|-----|------|
| 1 | O que é Aprendizado Estatístico — Y = f(X) + ε, Bias-Variance |
| 2 | Regressão Linear — coeficientes, premissas, métricas |
| 3 | Regressão Logística — odds ratio, threshold, métricas de classificação |
| 4 | Cross-Validation — todos os tipos com código |
| 5 | Regularização — Ridge, Lasso, Elastic Net |
| 6 | Árvores de Decisão e Random Forest |
| 7 | XGBoost — corrigindo o overfitting do Safra |
| 8 | SVM — normalização, C, Gamma, probabilidade |
| 9 | Clustering — K-means, Hierárquico, Silhouette |
| 10 | Redes Neurais e Funções de Ativação |
| 11 | MLflow — tracking de experimentos |
| 12 | Simulado Completo da Prova |

---

## Instalação das dependências

In [ ]:
# Execute esta célula primeiro
!pip install numpy pandas scikit-learn matplotlib scipy xgboost mlflow statsmodels --quiet

---
# DIA 1 — O que é Aprendizado Estatístico

## Y = f(X) + ε

- **Y** → o que você quer prever (NPS, inadimplência, churn)
- **X** → o que você observa (tempo de atendimento, segmento, produto)
- **f(X)** → a relação sistemática que o modelo aprende
- **ε** → erro irredutível — nenhum modelo elimina isso

**Regra de ouro:** o erro irredutível define o teto de qualquer modelo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

np.random.seed(42)

# Gerando dados com Y = f(X) + epsilon
X = np.linspace(0, 10, 100).reshape(-1, 1)
f_real = 2 * X.ravel() + np.sin(X.ravel())
epsilon = np.random.normal(0, 0.8, 100)  # erro irredutivel
Y = f_real + epsilon

modelo = LinearRegression().fit(X, Y)
Y_pred = modelo.predict(X)

plt.figure(figsize=(10, 4))
plt.scatter(X, Y, alpha=0.4, label='Y observado (f + ε)')
plt.plot(X, f_real, 'g-', linewidth=2, label='f(X) real')
plt.plot(X, Y_pred, 'r-', linewidth=2, label='f(X) estimado')
plt.legend()
plt.title('Y = f(X) + ε — o modelo aprende f, nunca elimina ε')
plt.tight_layout()
plt.show()

print(f'Erro irredutível (std dos resíduos): {np.std(Y - Y_pred):.3f}')
print(f'Desvio real do epsilon: 0.800')
print(f'→ O modelo não consegue ir abaixo do erro irredutível')

## Bias-Variance Tradeoff

```
Erro total = Bias² + Variância + Erro irredutível
```

- **Alto Bias (underfitting):** modelo simples demais, erra sistematicamente
- **Alta Variância (overfitting):** modelo complexo demais, memoriza ruído
- **Objetivo:** encontrar o equilíbrio

In [ ]:
np.random.seed(42)
X_data = np.sort(np.random.uniform(0, 10, 80)).reshape(-1, 1)
Y_data = np.sin(X_data.ravel()) + np.random.normal(0, 0.3, 80)

graus = [1, 3, 10, 20]
print(f"{'Grau':>5} | {'MSE Treino':>12} | {'MSE Val':>10} | {'Diagnóstico'}")
print("-" * 60)

for grau in graus:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=grau)),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_data, Y_data)
    mse_treino = np.mean((Y_data - pipe.predict(X_data))**2)
    scores = cross_val_score(pipe, X_data, Y_data, cv=5,
                             scoring='neg_mean_squared_error')
    mse_val = -scores.mean()

    if grau <= 2:
        diag = '← UNDERFITTING (alto bias)'
    elif grau <= 5:
        diag = '← PONTO ÓTIMO'
    else:
        diag = '← OVERFITTING (alta variância)'

    print(f"{grau:>5} | {mse_treino:>12.4f} | {mse_val:>10.4f} | {diag}")

---
# DIA 2 — Regressão Linear Completa

```
Y = β₀ + β₁X₁ + β₂X₂ + ... + βₚXₚ + ε
```

**β₁:** quanto Y muda para cada unidade de X₁, **mantendo todos os outros X constantes**.

### Métricas:
- **MSE:** penaliza erros grandes (ao quadrado)
- **RMSE:** mesma unidade de Y, penaliza erros grandes
- **MAE:** robusto a outliers
- **R²:** proporção da variância explicada — **NUNCA diminui com mais variáveis**
- **R² ajustado:** pode diminuir quando variável irrelevante é adicionada
- **MAPE:** INDEFINIDO quando y_true = 0

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

np.random.seed(42)
n = 500
tempo_atendimento = np.random.normal(10, 3, n)
tempo_resolucao = np.random.normal(5, 2, n)
nivel_credito = np.random.normal(7, 1.5, n)
epsilon = np.random.normal(0, 1, n)

nps = (8 - 0.3 * tempo_atendimento - 0.5 * tempo_resolucao
       + 0.4 * nivel_credito + epsilon)
nps = np.clip(nps, 0, 10)

df = pd.DataFrame({
    'tempo_atendimento': tempo_atendimento,
    'tempo_resolucao': tempo_resolucao,
    'nivel_credito': nivel_credito,
    'nps': nps
})

X = df[['tempo_atendimento', 'tempo_resolucao', 'nivel_credito']]
y = df['nps']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# sklearn para predição
modelo_sk = LinearRegression().fit(X_train, y_train)
y_pred = modelo_sk.predict(X_test)

print("=" * 50)
print("COEFICIENTES (sklearn)")
print("=" * 50)
for feat, coef in zip(X.columns, modelo_sk.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"  intercepto: {modelo_sk.intercept_:.4f}")

print("\n" + "=" * 50)
print("MÉTRICAS")
print("=" * 50)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
n_te = len(y_test)
p    = X.shape[1]
r2_adj = 1 - (1-r2)*(n_te-1)/(n_te-p-1)

print(f"  MSE:         {mse:.4f}")
print(f"  RMSE:        {rmse:.4f}")
print(f"  MAE:         {mae:.4f}")
print(f"  R²:          {r2:.4f}  ← NUNCA diminui com mais variáveis")
print(f"  R² ajustado: {r2_adj:.4f} ← pode diminuir")

print("\n" + "=" * 50)
print("INFERÊNCIA (statsmodels) — p-valores e intervalos")
print("=" * 50)
X_sm = sm.add_constant(X_train)
modelo_sm = sm.OLS(y_train, X_sm).fit()
print(modelo_sm.summary().tables[1])

In [ ]:
# PREMISSAS DA REGRESSÃO LINEAR — análise de resíduos
from scipy import stats

residuos = y_train.values - modelo_sk.predict(X_train)
y_fitted = modelo_sk.predict(X_train)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(y_fitted, residuos, alpha=0.4)
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_xlabel('Valores ajustados')
axes[0].set_ylabel('Resíduos')
axes[0].set_title('Resíduos vs Fitted\n(sem padrão = OK)')

stats.probplot(residuos, dist='norm', plot=axes[1])
axes[1].set_title('QQ Plot\n(diagonal = normalidade)')

axes[2].hist(residuos, bins=30, edgecolor='black')
axes[2].set_title('Distribuição dos resíduos')

plt.tight_layout()
plt.show()

print("PREMISSAS DA REGRESSÃO LINEAR:")
print("1. Linearidade — relação linear entre X e Y")
print("2. Independência — observações independentes")
print("3. Homocedasticidade — variância dos erros constante")
print("4. Normalidade dos resíduos — necessária para inferência")
print("5. Sem multicolinearidade — X's não altamente correlacionados")
print("")
print("ATENÇÃO: y = ax SEM intercepto → média dos resíduos ≠ 0 em geral!")
print("(Isso foi cobrado na Q15 da prova)")

---
# DIA 3 — Regressão Logística e Métricas de Classificação

## Por que não usar regressão linear para Y binário?
- Regressão linear pode prever valores < 0 e > 1 — sem sentido como probabilidade
- Regressão logística usa a sigmoide: sempre entre 0 e 1

```
log(p / 1-p) = β₀ + β₁X₁ + ...
p = sigmoid(β₀ + β₁X₁ + ...)
```

**Odds Ratio = e^β₁**: para cada unidade de X₁, as odds de Y=1 são multiplicadas por e^β₁

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                              roc_curve, f1_score, log_loss)

np.random.seed(42)
n = 1000

# Simulando detratores (20% da base — desbalanceada)
y_true = np.random.binomial(1, 0.2, n)
proba = np.where(y_true == 1,
                 np.random.beta(6, 3, n),
                 np.random.beta(3, 6, n))

threshold = 0.5
y_pred = (proba >= threshold).astype(int)

cm = confusion_matrix(y_true, y_pred)
TP = cm[1,1]; TN = cm[0,0]; FP = cm[0,1]; FN = cm[1,0]

print("=" * 55)
print("MATRIZ DE CONFUSÃO")
print("=" * 55)
print(f"          Previsto 0   Previsto 1")
print(f"Real 0:   TN={TN:4d}      FP={FP:4d}  ← Não detrator")
print(f"Real 1:   FN={FN:4d}      TP={TP:4d}  ← Detrator")

acuracia  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
auc       = roc_auc_score(y_true, proba)
logloss   = log_loss(y_true, proba)

print("\n" + "=" * 55)
print("MÉTRICAS — TODAS AS FÓRMULAS")
print("=" * 55)
print(f"  Acurácia  = (TP+TN)/(total)    = {acuracia:.4f}")
print(f"  ⚠ Com 80% classe 0: modelo idiota tem acurácia 0.80!")
print(f"  Precision = TP/(TP+FP)         = {precision:.4f}")
print(f"    Dos previstos DETRATOR, {precision*100:.1f}% realmente são")
print(f"  Recall    = TP/(TP+FN)         = {recall:.4f}")
print(f"    Dos DETRATORES reais, capturei {recall*100:.1f}%")
print(f"  F1        = 2*P*R/(P+R)        = {f1:.4f}")
print(f"  ROC-AUC   =                    = {auc:.4f}")
print(f"  Log Loss  =                    = {logloss:.4f}")

print("\n" + "=" * 55)
print("TRADEOFF PRECISION-RECALL por threshold")
print("=" * 55)
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    yp = (proba >= thresh).astype(int)
    tp = ((yp==1) & (y_true==1)).sum()
    fp = ((yp==1) & (y_true==0)).sum()
    fn = ((yp==0) & (y_true==1)).sum()
    p  = tp/(tp+fp) if (tp+fp)>0 else 0
    r  = tp/(tp+fn) if (tp+fn)>0 else 0
    print(f"  Thresh={thresh}: Precision={p:.3f} | Recall={r:.3f}")
print("")
print("Threshold BAIXO → mais recall (captura mais, mais FP)")
print("Threshold ALTO  → mais precision (menos FP, perde detratores)")

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_true, proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.3f})')
plt.plot([0,1], [0,1], 'k--', label='Modelo aleatório (AUC=0.5)')
plt.xlabel('FPR (Taxa de Falso Positivo)')
plt.ylabel('TPR (Recall)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Interpretação da AUC:")
print("  0.5  = aleatório, modelo inútil")
print("  0.7  = aceitável para muitos contextos bancários")
print("  0.8  = boa discriminação")
print("  >0.95 = SUSPEITO — provavelmente tem data leakage!")

---
# DIA 4 — Cross-Validation Completo

## O padrão da prova do Itaú
Confirmado com os dados reais: **`KFold(n_splits=K, shuffle=False)`**

⚠️ `neg_mean_squared_error` retorna **negativo** — multiplique por -1!

In [ ]:
from sklearn.model_selection import (KFold, StratifiedKFold, TimeSeriesSplit,
                                     cross_validate)
from sklearn.datasets import make_classification

X_cv, y_cv = make_classification(n_samples=500, n_features=10,
                                  weights=[0.8, 0.2], random_state=42)
modelo_cv = LogisticRegression(max_iter=1000)

print("=" * 70)
print("COMPARAÇÃO DE ESTRATÉGIAS DE CROSS-VALIDATION")
print("=" * 70)

estrategias = [
    ('KFold(5, shuffle=False) ← PADRÃO PROVA ITAÚ',
     KFold(n_splits=5, shuffle=False)),
    ('KFold(5, shuffle=True)',
     KFold(n_splits=5, shuffle=True, random_state=42)),
    ('StratifiedKFold(5) ← MELHOR para desbalanceado',
     StratifiedKFold(n_splits=5, shuffle=False)),
    ('TimeSeriesSplit(5) ← para dados temporais',
     TimeSeriesSplit(n_splits=5)),
]

for nome, cv in estrategias:
    res = cross_validate(modelo_cv, X_cv, y_cv, cv=cv,
                         scoring='roc_auc', return_train_score=True)
    tr = res['train_score'].mean()
    va = res['test_score'].mean()
    va_std = res['test_score'].std()
    print(f"\n{nome}")
    print(f"  Treino: {tr:.4f} | Val: {va:.4f} ± {va_std:.4f}")

print("\n" + "=" * 70)
print("ATENÇÃO — MSE NEGATIVO")
print("=" * 70)
res_mse = cross_validate(modelo_cv, X_cv, y_cv,
                          cv=KFold(5, shuffle=False),
                          scoring='neg_log_loss',
                          return_train_score=True)
print(f"  Valor bruto (negativo): {res_mse['test_score'].mean():.4f}")
print(f"  Valor correto (×-1):    {-res_mse['test_score'].mean():.4f}")
print(f"  ← Sempre multiplique por -1 ao usar neg_*")

---
# DIA 5 — Regularização: Ridge, Lasso, Elastic Net

| Método | Penalidade | Zera coef? | Quando usar |
|--------|-----------|-----------|-------------|
| Ridge | L2 = Σβ² | Não | Multicolinearidade, manter todas variáveis |
| Lasso | L1 = Σ\|β\| | Sim | Seleção automática de variáveis |
| Elastic Net | L1 + L2 | Sim (alguns) | Variáveis correlacionadas + seleção |

**Na Regressão Logística:** `C = 1/alpha` (C pequeno = regularização FORTE)

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n, p = 300, 20
X_reg = np.random.randn(n, p)
beta_real = np.zeros(p)
beta_real[:5] = [3, -2, 1.5, -1, 0.8]  # só 5 das 20 importam
y_reg = X_reg @ beta_real + np.random.randn(n) * 0.5

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reg)

modelos_reg = {
    'Linear (sem regularização)': LinearRegression(),
    'Ridge (L2, alpha=1.0)':      Ridge(alpha=1.0),
    'Lasso (L1, alpha=0.1)':      Lasso(alpha=0.1),
    'Elastic Net (a=0.1, r=0.5)': ElasticNet(alpha=0.1, l1_ratio=0.5),
}

print(f"{'Modelo':<32} {'Coefs=0':>8} {'Coefs≠0':>8}")
print("-" * 52)
for nome, mod in modelos_reg.items():
    mod.fit(X_scaled, y_reg)
    zeros  = (np.abs(mod.coef_) < 1e-6).sum()
    nzeros = p - zeros
    print(f"{nome:<32} {zeros:>8} {nzeros:>8}")

print("\nCONCLUSÕES:")
print("Ridge: encolhe todos, NUNCA zera → mantém todas as variáveis")
print("Lasso: zera irrelevantes → seleção automática de variáveis")
print("ElasticNet: mix — melhor quando há variáveis correlacionadas")
print("")
print("Lambda=0 em Ridge   → REGRESSÃO LINEAR PURA (sem penalidade)")
print("Lambda→∞ em Ridge   → coeficientes → 0 (NUNCA → infinito!)")
print("")
print("C no sklearn LogisticRegression:")
print("  C = 1 / alpha")
print("  C=0.001 → alpha=1000 → regularização MUITO FORTE")
print("  C=10    → alpha=0.1  → regularização fraca")

In [ ]:
# Caminho dos coeficientes em função de alpha
alphas = np.logspace(-3, 3, 100)
coefs_ridge = [Ridge(alpha=a).fit(X_scaled, y_reg).coef_ for a in alphas]
coefs_lasso = [Lasso(alpha=a, max_iter=5000).fit(X_scaled, y_reg).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i in range(p):
    cor = 'blue' if i < 5 else 'lightgray'
    axes[0].plot(alphas, [c[i] for c in coefs_ridge], color=cor, alpha=0.7)
    axes[1].plot(alphas, [c[i] for c in coefs_lasso], color=cor, alpha=0.7)

for ax, titulo in zip(axes, ['Ridge (L2): coefs → 0 mas NUNCA zerados',
                               'Lasso (L1): coefs zerados exatamente']):
    ax.set_xscale('log')
    ax.set_xlabel('Alpha (lambda)')
    ax.set_ylabel('Coeficiente')
    ax.set_title(titulo + '\n(azul = variáveis reais)')

plt.tight_layout()
plt.show()

---
# DIA 6 — Árvores de Decisão e Random Forest

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X_tree, y_tree = make_classification(n_samples=1000, n_features=10,
                                      n_informative=5, random_state=42)
kf5 = KFold(n_splits=5, shuffle=False)

print("=" * 65)
print("ÁRVORE — EFEITO DA PROFUNDIDADE (criterion=entropy)")
print("=" * 65)
print(f"{'max_depth':>10} | {'Treino LL':>10} | {'Val LL':>10} | {'Gap':>8} | Diagnóstico")
print("-" * 75)

for max_d in [None, 2, 5, 10, 20]:
    arvore = DecisionTreeClassifier(criterion='entropy',
                                    max_depth=max_d, random_state=42)
    res = cross_validate(arvore, X_tree, y_tree, cv=kf5,
                         scoring='neg_log_loss', return_train_score=True)
    tr = -res['train_score'].mean()
    va = -res['test_score'].mean()
    diag = '← overfitting total' if max_d is None else ''
    print(f"{str(max_d):>10} | {tr:>10.4f} | {va:>10.4f} | {va-tr:>8.4f} | {diag}")

print("\n⚠ Sem poda: Log Loss treino = 0.0 → overfitting total")
print("⚠ Log Loss É válida para classificação (Q20 da prova!)")

print("\n" + "=" * 65)
print("RANDOM FOREST — O QUE CAUSA OVERFITTING")
print("=" * 65)

configs = [
    (10,  None, '10 árvores, sem poda'),
    (100, None, '100 árvores, sem poda'),
    (100, 5,    '100 árvores, max_depth=5'),
    (500, 5,    '500 árvores, max_depth=5'),
]
for n_est, max_d, label in configs:
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42)
    res = cross_validate(rf, X_tree, y_tree, cv=kf5,
                         scoring='roc_auc', return_train_score=True)
    tr = res['train_score'].mean()
    va = res['test_score'].mean()
    print(f"  {label:<35} Treino={tr:.4f} | Val={va:.4f} | Gap={tr-va:.4f}")

print("\nCONCLUSÕES RF:")
print("  - Mais árvores: NÃO causa overfitting, estabiliza")
print("  - Árvores profundas: CAUSA overfitting")
print("  - Random Forest NÃO tem learning_rate (isso é GBM/XGBoost!)")

---
# DIA 7 — XGBoost: Corrigindo o Overfitting do Safra

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

X_xgb, y_xgb = make_classification(n_samples=1000, n_features=15,
                                    n_informative=8, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_xgb, y_xgb,
                                             test_size=0.2, random_state=42)

print("=" * 60)
print("O QUE CAUSOU SEU OVERFITTING NO SAFRA")
print("=" * 60)

# Modelo que overfita
xgb_ruim = XGBClassifier(n_estimators=500, learning_rate=0.3,
                          max_depth=10, verbosity=0, random_state=42)
xgb_ruim.fit(X_tr, y_tr)

auc_tr1 = roc_auc_score(y_tr,  xgb_ruim.predict_proba(X_tr)[:,1])
auc_va1 = roc_auc_score(y_val, xgb_ruim.predict_proba(X_val)[:,1])
print(f"\nXGBoost MAL configurado (como no Safra):")
print(f"  AUC treino: {auc_tr1:.4f}")
print(f"  AUC val:    {auc_va1:.4f}")
print(f"  Gap: {auc_tr1-auc_va1:.4f} ← overfitting")

# Modelo correto com early stopping
xgb_bom = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,       # lr BAIXO
    max_depth=4,              # profundidade CONTROLADA
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,           # regularização L2
    reg_alpha=0.1,            # regularização L1
    eval_metric='auc',
    early_stopping_rounds=50, # PARA quando val para de melhorar
    verbosity=0,
    random_state=42
)
xgb_bom.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

auc_tr2 = roc_auc_score(y_tr,  xgb_bom.predict_proba(X_tr)[:,1])
auc_va2 = roc_auc_score(y_val, xgb_bom.predict_proba(X_val)[:,1])
print(f"\nXGBoost BEM configurado (com early stopping):")
print(f"  AUC treino: {auc_tr2:.4f}")
print(f"  AUC val:    {auc_va2:.4f}")
print(f"  Gap: {auc_tr2-auc_va2:.4f} ← muito menor")
print(f"  Árvores utilizadas: {xgb_bom.best_iteration} de 1000")

print("\n" + "=" * 60)
print("RF vs XGBoost — DIFERENÇAS CRÍTICAS")
print("=" * 60)
diferencas = [
    ('Construção',          'Paralela',      'Sequencial'),
    ('Tem learning_rate',   'NÃO',           'SIM'),
    ('Early stopping',      'NÃO',           'SIM'),
    ('Overfitting via',     'max_depth',     'lr alto + n_trees'),
    ('Regularização L1/L2', 'NÃO nativa',    'SIM (built-in)'),
]
print(f"{'Característica':<25} {'Random Forest':>15} {'XGBoost':>15}")
print("-" * 58)
for item in diferencas:
    print(f"{item[0]:<25} {item[1]:>15} {item[2]:>15}")

---
# DIA 8 — SVM Completo

In [ ]:
from sklearn.svm import SVC, SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_svm, y_svm = make_classification(n_samples=500, n_features=10, random_state=42)

print("=" * 60)
print("1. SVM SEMPRE PRECISA DE NORMALIZAÇÃO")
print("=" * 60)

svm_sem = SVC(kernel='rbf', C=1.0, probability=True)
res_sem = cross_validate(svm_sem, X_svm, y_svm, cv=KFold(5, shuffle=False),
                          scoring='roc_auc')

pipe_com = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=1.0, probability=True))])
res_com = cross_validate(pipe_com, X_svm, y_svm, cv=KFold(5, shuffle=False),
                          scoring='roc_auc')

print(f"  Sem normalização: AUC = {res_sem['test_score'].mean():.4f}")
print(f"  Com normalização: AUC = {res_com['test_score'].mean():.4f}")
print(f"  → A Q29 disse 'não é necessário normalizar' — FALSO")

print("\n" + "=" * 60)
print("2. EFEITO DO PARÂMETRO C")
print("=" * 60)
print(f"{'C':>8} | {'Treino':>8} | {'Val':>8} | {'Gap':>8}")
print("-" * 38)
for C in [0.001, 0.1, 1.0, 10.0, 100.0]:
    pipe = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=C, probability=True))])
    res = cross_validate(pipe, X_svm, y_svm, cv=KFold(5, shuffle=False),
                         scoring='roc_auc', return_train_score=True)
    tr = res['train_score'].mean()
    va = res['test_score'].mean()
    print(f"{C:>8.3f} | {tr:>8.4f} | {va:>8.4f} | {tr-va:>8.4f}")

print("  C pequeno → regularização forte → modelo simples")
print("  C grande  → menos regularização → overfitting")

print("\n" + "=" * 60)
print("3. EFEITO DO GAMMA (kernel RBF)")
print("=" * 60)
print(f"{'Gamma':>8} | {'Treino':>8} | {'Val':>8}")
print("-" * 30)
for gamma in [0.001, 0.01, 0.1, 1.0, 10.0]:
    pipe = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=1.0,
                                 gamma=gamma, probability=True))])
    res = cross_validate(pipe, X_svm, y_svm, cv=KFold(5, shuffle=False),
                         scoring='roc_auc', return_train_score=True)
    print(f"{gamma:>8.3f} | {res['train_score'].mean():>8.4f} | {res['test_score'].mean():>8.4f}")
print("  Gamma alto → raio pequeno → overfitting")
print("  Gamma baixo → raio grande → underfitting")
print("")
print("4. SVM E PROBABILIDADE:")
print("   probability=True usa Platt Scaling internamente")
print("   SVM NÃO produz probabilidade nativa — diferente da Logística")
print("   Logística: probabilidade CALIBRADA por construção")

---
# DIA 9 — Clustering Completo

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram

np.random.seed(42)
n_pc = 100
clusters_data = np.vstack([
    np.random.normal([2, 2], 0.5, (n_pc, 2)),
    np.random.normal([8, 8], 0.5, (n_pc, 2)),
    np.random.normal([2, 8], 0.5, (n_pc, 2)),
    np.random.normal([8, 2], 0.5, (n_pc, 2)),
])

# Método do Cotovelo
inertias = []
silhouettes = []
Ks = range(2, 10)

for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(clusters_data)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(clusters_data, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(Ks, inertias, 'bo-')
axes[0].axvline(4, color='red', linestyle='--', label='K=4 ótimo')
axes[0].set_title('Método do Cotovelo\n(procure o joelho)')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inércia')
axes[0].legend()

axes[1].plot(Ks, silhouettes, 'go-')
axes[1].axvline(4, color='red', linestyle='--', label='K=4 ótimo')
axes[1].set_title('Silhouette Score\n(maior = melhor)')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].legend()

plt.tight_layout()
plt.show()

print("=" * 65)
print("TODOS OS LINKAGES — COMPARAÇÃO")
print("=" * 65)
linkages_dict = {
    'single':   'Distância MÍNIMA — sensível a outliers',
    'complete': 'Distância MÁXIMA — menos sensível (Q32 da prova)',
    'average':  'MÉDIA par-a-par — NÃO é centróide (Q31 da prova)',
    'ward':     'Minimiza variância — clusters equilibrados',
}

for method, desc in linkages_dict.items():
    Z = linkage(clusters_data, method=method)
    labels = fcluster(Z, 4, criterion='maxclust')
    sil = silhouette_score(clusters_data, labels)
    print(f"\n{method.upper():10s}: {desc}")
    print(f"           Silhouette com 4 grupos: {sil:.4f}")

In [ ]:
# Q30 DA PROVA — reproduzindo o resultado correto
import os

agrup_path = 'agrupamento.csv'  # coloque o arquivo na mesma pasta do notebook

if os.path.exists(agrup_path):
    df_agrup = pd.read_csv(agrup_path)
    Z_prova = linkage(df_agrup.values, method='single')

    print("=" * 55)
    print("Q30 DA PROVA — Single Linkage, 4 grupos")
    print("=" * 55)
    for d in sorted(set(Z_prova[:,2])):
        n_grupos = len(set(fcluster(Z_prova, d, criterion='distance')))
        if 3 <= n_grupos <= 5:
            print(f"  Limiar = {d:.4f} → {n_grupos} grupos")
    print("")
    print("Para 4 grupos: 2.21 < limiar < 4.63")
    print("O candidato marcou '0.5 < d < 2.2' — ERRADO")
    print("Resposta correta: faixa que inicia em 2.21")
else:
    print("Coloque agrupamento.csv na mesma pasta para reproduzir Q30")

print("\n" + "=" * 55)
print("NORMALIZAÇÃO EM CLUSTERING")
print("=" * 55)
print("K-means usa distância Euclidiana → SENSÍVEL à escala")
print("Variável com maior escala DOMINA a distância")
print("SEMPRE normalizar antes de K-means")
print("A Q36 disse 'clustering é invariante à normalização' → ERRADO")

---
# DIA 10 — Redes Neurais e Funções de Ativação

| Função | Output | Negativo? | Uso |
|--------|--------|-----------|-----|
| ReLU | ≥ 0 | **Nunca** | Camadas ocultas (padrão) |
| Leaky ReLU | Qualquer | **Pode** | Evita dying ReLU |
| Sigmoid | (0,1) | **Nunca** | Saída binária |
| Tanh | (-1,1) | **Pode** | Camadas ocultas |
| Linear | Qualquer | Pode | Regressão (saída) |
| Softmax | (0,1) soma=1 | Nunca | Saída multiclasse |

In [ ]:
x_ativ = np.linspace(-5, 5, 300)

funcoes_ativ = {
    'ReLU\n(output ≥ 0, nunca negativo)':       lambda x: np.maximum(0, x),
    'Leaky ReLU\n(PODE ser negativo)':           lambda x: np.where(x > 0, x, 0.01 * x),
    'Sigmoid\n(0 a 1, nunca negativo)':          lambda x: 1 / (1 + np.exp(-x)),
    'Tanh\n(-1 a 1, pode ser negativo)':         lambda x: np.tanh(x),
    'Linear\n(qualquer valor)':                   lambda x: x,
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (nome, func) in zip(axes, funcoes_ativ.items()):
    y_ativ = func(x_ativ)
    ax.plot(x_ativ, y_ativ, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(nome, fontsize=9)
    ax.set_ylim(-2, 2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("QUESTÃO Q17 DA PROVA:")
print("Saída de -0.001 — qual função NÃO pode gerar?")
print("  ReLU: max(0,x) → NUNCA negativo ✓ (resposta correta)")
print("  Sigmoid: entre 0 e 1 → NUNCA negativo ✓")
print("  Leaky ReLU: 0.01x para x<0 → PODE ser negativo ✗")
print("  O candidato marcou Leaky ReLU como errada — estava CERTO")
print("  Mas Leaky ReLU CAN produzir negativos — contradição no gabarito")
print("")
print("QUESTÃO Q28 DA PROVA:")
print("Rede com ativações LINEARES em todas as camadas:")
print("  Composição de funções lineares = UMA função linear")
print("  100 camadas lineares = mesma capacidade que 1 camada")
print("  Profundidade SÓ ajuda com funções NÃO-LINEARES")

---
# DIA 11 — MLflow: Tracking de Experimentos

In [ ]:
import mlflow
import mlflow.sklearn

X_mlf, y_mlf = make_classification(n_samples=1000, n_features=15,
                                    n_informative=8, random_state=42)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_mlf, y_mlf, test_size=0.2, random_state=42)

mlflow.set_experiment('guia_cientista_dados_itau')

modelos_mlf = [
    {
        'nome': 'LogReg_C0.1',
        'modelo': Pipeline([('sc', StandardScaler()),
                            ('lr', LogisticRegression(C=0.1, max_iter=1000))]),
        'params': {'modelo': 'LogisticRegression', 'C': 0.1}
    },
    {
        'nome': 'RF_depth5',
        'modelo': RandomForestClassifier(n_estimators=100,
                                         max_depth=5, random_state=42),
        'params': {'modelo': 'RandomForest', 'max_depth': 5}
    },
    {
        'nome': 'XGB_lr0.05',
        'modelo': XGBClassifier(n_estimators=200, learning_rate=0.05,
                                max_depth=4, verbosity=0, random_state=42),
        'params': {'modelo': 'XGBoost', 'learning_rate': 0.05}
    },
]

kf_mlf = KFold(n_splits=5, shuffle=False)
resultados_mlf = []

for cfg in modelos_mlf:
    with mlflow.start_run(run_name=cfg['nome']):

        mlflow.log_params(cfg['params'])

        res = cross_validate(cfg['modelo'], X_tr_m, y_tr_m,
                             cv=kf_mlf, scoring='roc_auc',
                             return_train_score=True)
        auc_tr = res['train_score'].mean()
        auc_va = res['test_score'].mean()

        cfg['modelo'].fit(X_tr_m, y_tr_m)
        probas = cfg['modelo'].predict_proba(X_te_m)[:,1]
        auc_te = roc_auc_score(y_te_m, probas)

        mlflow.log_metric('auc_treino', round(auc_tr, 4))
        mlflow.log_metric('auc_val_cv', round(auc_va, 4))
        mlflow.log_metric('auc_teste',  round(auc_te, 4))
        mlflow.log_metric('gap',        round(auc_tr - auc_va, 4))
        mlflow.sklearn.log_model(cfg['modelo'], 'modelo')

        resultados_mlf.append({
            'Modelo':      cfg['nome'],
            'AUC Treino':  round(auc_tr, 4),
            'AUC Val CV':  round(auc_va, 4),
            'AUC Teste':   round(auc_te, 4),
            'Gap':         round(auc_tr - auc_va, 4)
        })

print("RESULTADOS COMPARATIVOS")
print(pd.DataFrame(resultados_mlf).to_string(index=False))
print("\nPara visualizar no MLflow UI:")
print("  mlflow ui   (no terminal)")
print("  Abra: http://localhost:5000")

---
# DIA 12 — Simulado Completo da Prova

Coloque os arquivos CSV na mesma pasta do notebook:
- `regressao_Q1.csv`
- `regressao_Q2.csv`
- `classificacao_Q1.csv`
- `classificacao_Q2.csv`
- `agrupamento.csv`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier
from scipy.cluster.hierarchy import linkage, fcluster

print("=" * 65)
print("SIMULADO — REPRODUZINDO A PROVA DE 2019")
print("=" * 65)

arquivos = {
    'regressao_Q1':    'regressao_Q1.csv',
    'regressao_Q2':    'regressao_Q2.csv',
    'classificacao_Q1':'classificacao_Q1.csv',
    'classificacao_Q2':'classificacao_Q2.csv',
    'agrupamento':     'agrupamento.csv',
}

dados = {}
for nome, arquivo in arquivos.items():
    if os.path.exists(arquivo):
        dados[nome] = pd.read_csv(arquivo)
        print(f"  ✓ {arquivo} carregado — {dados[nome].shape}")
    else:
        print(f"  ✗ {arquivo} não encontrado — coloque na pasta do notebook")

kf5  = KFold(n_splits=5,  shuffle=False)
kf10 = KFold(n_splits=10, shuffle=False)

In [ ]:
# Q10 — Elastic Net
if 'regressao_Q1' in dados:
    df = dados['regressao_Q1']
    X, y = df.drop('target', axis=1), df['target']
    model = ElasticNet(alpha=1.0, l1_ratio=0.01)
    res = cross_validate(model, X, y, cv=kf5,
                         scoring='neg_mean_squared_error',
                         return_train_score=True)
    print(f"Q10 — Elastic Net")
    print(f"  Treino MSE: {-res['train_score'].mean():.4f}")
    print(f"  Val MSE:   {-res['test_score'].mean():.4f}")
    print(f"  → Esperado: ~0.2686 e ~0.2693")
    print(f"  → Padrão: KFold(5, shuffle=False) confirmado!")

In [ ]:
# Q11 — SVR
if 'regressao_Q2' in dados:
    df2 = dados['regressao_Q2']
    X2, y2 = df2.drop('target', axis=1), df2['target']
    model2 = SVR(kernel='linear', C=0.001)
    res2 = cross_validate(model2, X2, y2, cv=kf5,
                          scoring='neg_mean_squared_error',
                          return_train_score=True)
    print(f"Q11 — SVR (kernel=linear, C=0.001)")
    print(f"  Treino MSE: {-res2['train_score'].mean():.0f}")
    print(f"  Val MSE:   {-res2['test_score'].mean():.0f}")
    print(f"  → Esperado: ~20199 e ~20207")

In [ ]:
# Q20 — Árvore com Log Loss
if 'classificacao_Q1' in dados:
    df3 = dados['classificacao_Q1']
    X3, y3 = df3.drop('target', axis=1), df3['target']
    model3 = DecisionTreeClassifier(criterion='entropy')  # SEM poda!
    res3 = cross_validate(model3, X3, y3, cv=kf10,
                          scoring='neg_log_loss',
                          return_train_score=True)
    print(f"Q20 — Árvore (entropy, sem poda, 10 folds)")
    print(f"  Treino LogLoss: {-res3['train_score'].mean():.4f}  ← overfitting total")
    print(f"  Val LogLoss:   {-res3['test_score'].mean():.4f}")
    print(f"  → Candidato errou! Resposta correta: '0 e ~3'")
    print(f"  → Log Loss É válida para classificação")

In [ ]:
# Q21 — Logística com AUC
if 'classificacao_Q2' in dados:
    df4 = dados['classificacao_Q2']
    X4, y4 = df4.drop('target', axis=1), df4['target']
    model4 = LogisticRegression(C=0.1, max_iter=1000)  # L2, C=0.1
    res4 = cross_validate(model4, X4, y4, cv=kf10,
                          scoring='roc_auc',
                          return_train_score=True)
    print(f"Q21 — Logística (L2, C=0.1, 10 folds)")
    print(f"  Treino AUC: {res4['train_score'].mean():.4f}")
    print(f"  Val AUC:   {res4['test_score'].mean():.4f}")
    print(f"  → Candidato errou! Marcou 0.9/0.5 (overfitting)")
    print(f"  → Modelo bem regularizado — treino ≈ val")

In [ ]:
# Q30 — Single Linkage, 4 grupos
if 'agrupamento' in dados:
    df5 = dados['agrupamento']
    Z30 = linkage(df5.values, method='single')
    print("Q30 — Single Linkage, 4 grupos")
    print("Distâncias que mudam o número de grupos:")
    for d in sorted(set(Z30[:,2])):
        n_grupos = len(set(fcluster(Z30, d, criterion='distance')))
        if 3 <= n_grupos <= 5:
            print(f"  Limiar = {d:.4f} → {n_grupos} grupos")
    print("→ Para 4 grupos: limiar entre 2.21 e 4.63")
    print("→ Candidato errou! Marcou '0.5 < d < 2.2'")

---
# COLA RÁPIDA — O que todo Cientista de Dados sabe sem pensar

## Quando normalizar
| Normalizar? | Algoritmos |
|-------------|------------|
| **SEMPRE** | SVM, KNN, Regressão com Ridge/Lasso/ElasticNet, K-means |
| **NUNCA precisa** | Árvores, Random Forest, XGBoost, Gradient Boosting |

## Métricas por tipo de problema
| Problema | Métricas |
|----------|----------|
| Regressão | RMSE, MAE, R², R² ajustado |
| Classificação | ROC-AUC, F1, Precision, Recall, Log Loss |
| **NUNCA** | Acurácia com base desbalanceada; MAE para classificação; MAPE com y=0 |

## Regularização
- **Ridge:** lambda→0 = regressão pura. Lambda→∞ = coefs→0 (NUNCA infinito)
- **Lasso:** zera coefs. Instável com multicolinearidade
- **C no sklearn:** C = 1/alpha. **C pequeno = regularização forte**

## Cross-Validation — padrão do Itaú
```python
KFold(n_splits=K, shuffle=False)  # ESTE é o padrão da prova
-res['train_score'].mean()         # multiplica por -1!
```

## Funções de ativação
- **ReLU:** ≥ 0, NUNCA negativo
- **Leaky ReLU:** PODE ser negativo
- **Sigmoid:** entre 0 e 1, NUNCA negativo
- **Linear em todas as camadas:** rede colapsa para transformação linear

## Clustering
- **Single linkage:** mínima distância, sensível a outliers
- **Complete linkage:** máxima distância, menos sensível
- **Average linkage:** média par-a-par (NÃO é centróide!)
- **Ward:** minimiza variância — mais equilibrado

## Calibração de probabilidade
- **Logística:** calibrada por construção ✓
- **Random Forest/Árvores:** NÃO calibradas
- **SVM:** usa Platt Scaling com probability=True

In [ ]:
# TREINO DE VELOCIDADE — responda mentalmente antes de ver a resposta

qa = [
    ("Qual a diferença entre bias e variância?",
     "Bias: erro sistemático, modelo simples. Variância: sensibilidade ao treino."),
    ("Quando usar Lasso vs Ridge?",
     "Lasso: seleção de variáveis, zera coefs. Ridge: mantém todos, ótimo com multicolinearidade."),
    ("O que acontece com lambda→∞ no Ridge?",
     "Coeficientes → 0. NUNCA → infinito."),
    ("Por que acurácia é ruim com base desbalanceada?",
     "Modelo idiota (prevê sempre a classe majoritária) tem acurácia alta sem discriminar."),
    ("O que é data leakage?",
     "Informação do futuro/target vaza para o treino. Detectar: AUC muito alta, feature importance inesperada."),
    ("Qual CV usar para dados temporais?",
     "TimeSeriesSplit — treina no passado, valida no futuro. Nunca KFold com shuffle."),
    ("Qual função de ativação nunca gera negativos?",
     "ReLU: max(0,x). Sigmoid: entre 0 e 1."),
    ("Por que RF não overfita com muitas árvores?",
     "Mais árvores = mais estável. Overfitting vem de max_depth, não de n_estimators."),
    ("Qual linkage é menos sensível a outliers?",
     "Complete linkage — usa distância máxima. Outlier não cria bridge entre clusters."),
    ("O que significa AUC = 0.5?",
     "Modelo aleatório. Não discrimina as classes melhor que chance."),
    ("Quando usar F1 em vez de acurácia?",
     "Quando a base é desbalanceada. F1 equilibra precision e recall."),
    ("O que é o parâmetro C no SVM?",
     "C = 1/alpha. C pequeno = regularização forte. C grande = modelo mais complexo."),
    ("Diferença entre precision e recall?",
     "Precision: dos previstos positivos, quantos são reais? Recall: dos reais, quantos capturei?"),
    ("Por que normalizar antes do K-means?",
     "K-means usa distância Euclidiana. Variável com maior escala domina o resultado."),
    ("O que é silhouette score?",
     "Mede coesão interna vs separação entre clusters. -1 a 1. Maior = melhor."),
    ("Por que árvore sem poda tem Log Loss treino = 0?",
     "Sem restrição, memoriza cada exemplo. P=1.0 para a classe correta no treino."),
    ("O que é Platt Scaling?",
     "Método para calibrar probabilidades de SVM. probability=True no sklearn usa isso."),
    ("Qual métrica para regressão com outliers?",
     "MAE — erro absoluto médio. Não penaliza outliers ao quadrado como RMSE."),
    ("O que é CV estratificado?",
     "Garante que cada fold tem a mesma proporção de classes que o dataset original."),
    ("Principal diferença XGBoost vs Random Forest?",
     "RF: paralelo, sem learning_rate. XGBoost: sequencial, tem lr, precisa early stopping."),
]

print("TREINO DE VELOCIDADE — 20 QUESTÕES")
print("Cubra as respostas e responda mentalmente antes de ver")
print("=" * 65)
for i, (pergunta, resposta) in enumerate(qa, 1):
    print(f"\nQ{i:02d}: {pergunta}")
    print(f"  → {resposta}")

---
# FIM DO GUIA

## Próximos passos

1. **Dias 1-12:** execute cada célula e entenda o output antes de avançar
2. **Semanas 3-8:** aplique tudo no projeto de CNPJ
3. **Mês 2-3:** MLflow completo + portfólio no GitHub
4. **Candidatura:** Novembro 2025

## Arquivos necessários para o simulado
Coloque na mesma pasta do notebook:
- `regressao_Q1.csv`
- `regressao_Q2.csv`  
- `classificacao_Q1.csv`
- `classificacao_Q2.csv`
- `agrupamento.csv`